# Verifying Circuit Equivalence (Section 4.1)

This notebook illustrates the **circuit-equivalence verification** application of
[arXiv:2603.24717](https://arxiv.org/abs/2603.24717), Section 4.1, using the Python
`PhasedOutcomeCompleteSimulation` API. The companion notebook *Verifying Symbolic-Rotation Circuits*
covers the same idea for **channels** (via Choi states); here we focus on the literal Section 4.1
construction for **state preparation**.

## The reduction

We want to decide whether two parameterized state-preparation circuits agree for *every* rotation
angle $\alpha$:
$$ C_1\, e^{i\alpha Z}\, C_2\,|0\cdots0\rangle \;=\; D_1\, e^{i\alpha Z}\, D_2\,|0\cdots0\rangle \quad\text{for all }\alpha. $$
Section 4.1 shows this is equivalent to a *single* **exact** stabilizer-state equality, with the
continuous angle replaced by a binary symbolic exponent $a$:
$$ C_1\, Z^{a}\, C_2\,|0\cdots0\rangle \;=\; D_1\, Z^{a}\, D_2\,|0\cdots0\rangle. $$
Exactness is the whole point: the equality must hold *including* the relative phase between the
$a=0$ and $a=1$ branches, since that phase is what encodes the rotation for every $\alpha$. An
ordinary, phase-blind stabilizer comparison is not enough.

There is **no dedicated verification function**: the check is simply `phased_action(...)` followed by
`PhasedCircuitAction.is_equivalent(...)`, exactly mirroring how `OutcomeCompleteSimulation` performs
phaseless equality checking. The symbolic exponent $Z^{a}$ is an `allocate_symbolic_angle()` bit
driving a conditional $Z$.

## Setup

In [1]:
from paulimer import (
    PhasedOutcomeCompleteSimulation,
    PhasedCircuitAction,
    SparsePauli,
    UnitaryOpcode,
)


def prepared_state_action(qubit_count, build):
    """Record a state-preparation circuit C_1 (prod_k Z_k^{a_k}) C_2 |0...0> as a phased action.

    State preparation is the `inputs = []` case of `phased_action`; the outputs are all qubits.
    """
    sim = PhasedOutcomeCompleteSimulation(qubit_count)
    build(sim)
    return sim.phased_action([], list(range(qubit_count)))


def prepare_plus(sim, qubit_count):
    """Prepare |+...+> by Hadamarding every qubit."""
    for qubit in range(qubit_count):
        sim.apply_unitary(UnitaryOpcode.Hadamard, [qubit])

## Two factorizations of the same state

A clean Section 4.1 instance: the parameterized state $e^{i\alpha Z_0 Z_1}\,|{+}{+}\rangle$ written
two different ways. Conjugating a single-qubit rotation by a `CNOT` turns it into a two-qubit $ZZ$
rotation, because $\mathrm{CNOT}_{01}\,Z_1\,\mathrm{CNOT}_{01} = Z_0 Z_1$ and
$\mathrm{CNOT}_{01}\,|{+}{+}\rangle = |{+}{+}\rangle$:
$$ e^{i\alpha Z_0 Z_1}\,|{+}{+}\rangle \;=\; \mathrm{CNOT}_{01}\; e^{i\alpha Z_1}\; \mathrm{CNOT}_{01}\,|{+}{+}\rangle. $$
The two circuits are different Clifford factorizations $C_1 Z^a C_2$ of the same parameterized state,
so they must verify as equivalent.

In [2]:
def direct_zz(sim):                  # e^{iα Z0Z1} |++>   (C_2 = H⊗H, Z^a on Z0Z1, C_1 = I)
    prepare_plus(sim, 2)
    a = sim.allocate_symbolic_angle()
    sim.apply_conditional_pauli(SparsePauli("Z_0 Z_1"), [a])


def conjugated_zz(sim):              # CNOT01 · e^{iα Z1} · CNOT01 |++>
    prepare_plus(sim, 2)
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
    a = sim.allocate_symbolic_angle()
    sim.apply_conditional_pauli(SparsePauli("Z_1"), [a])
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])


direct = prepared_state_action(2, direct_zz)
conjugated = prepared_state_action(2, conjugated_zz)

assert direct.is_equivalent(conjugated)
assert conjugated.is_equivalent(direct)            # verification is symmetric
print("verified equal:  e^{iα Z0Z1}|++>  ==  CNOT01 · e^{iα Z1} · CNOT01 |++>")

verified equal:  e^{iα Z0Z1}|++>  ==  CNOT01 · e^{iα Z1} · CNOT01 |++>


## A purely-phase difference

The verification is phase-sensitive. The states $e^{+i\alpha Z}\,|{+}\rangle$ and
$e^{-i\alpha Z}\,|{+}\rangle$ have **identical** stabilizer (symplectic) data, differing only by a
relative $-1$ on the $a=1$ branch. A phase-blind comparison (`is_equivalent_up_to_signs`) reports them
equal; the phase-aware `is_equivalent` correctly separates them.

In [3]:
def rotate_plus(sign):
    pauli = SparsePauli("Z_0") if sign > 0 else SparsePauli("-Z_0")

    def build(sim):
        prepare_plus(sim, 1)
        a = sim.allocate_symbolic_angle()
        sim.apply_conditional_pauli(pauli, [a])

    return prepared_state_action(1, build)


positive, negative = rotate_plus(+1), rotate_plus(-1)

assert positive.is_equivalent_up_to_signs(negative)   # phase-blind: indistinguishable
assert not positive.is_equivalent(negative)           # phase-aware: distinct
print("phase-blind:  e^{+iα Z}|+>  and  e^{-iα Z}|+>  are INDISTINGUISHABLE")
print("phase-aware:  e^{+iα Z}|+>  !=  e^{-iα Z}|+>")

phase-blind:  e^{+iα Z}|+>  and  e^{-iα Z}|+>  are INDISTINGUISHABLE
phase-aware:  e^{+iα Z}|+>  !=  e^{-iα Z}|+>


## Several independent angles

The reduction generalizes to several independent symbolic angles at once. We verify
$e^{i\alpha Z_0 Z_1}\, e^{i\beta Z_0}\,|{+}{+}\rangle$ against its CNOT-conjugated factorization. The
two angles are allocated in the **same order** in both circuits, so the virtual-angle bits correspond
one-to-one. Negating the second rotation's Pauli injects a pure branch-phase difference, which is
detected.

In [4]:
def two_angle_direct(negate_second):
    second = SparsePauli("-Z_0") if negate_second else SparsePauli("Z_0")

    def build(sim):
        prepare_plus(sim, 2)
        a = sim.allocate_symbolic_angle()              # α  (allocated first)
        sim.apply_conditional_pauli(SparsePauli("Z_0 Z_1"), [a])
        b = sim.allocate_symbolic_angle()              # β  (allocated second)
        sim.apply_conditional_pauli(second, [b])

    return prepared_state_action(2, build)


def two_angle_conjugated(sim):
    prepare_plus(sim, 2)
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
    a = sim.allocate_symbolic_angle()                  # α
    sim.apply_conditional_pauli(SparsePauli("Z_1"), [a])
    sim.apply_unitary(UnitaryOpcode.ControlledX, [0, 1])
    b = sim.allocate_symbolic_angle()                  # β
    sim.apply_conditional_pauli(SparsePauli("Z_0"), [b])


conjugated = prepared_state_action(2, two_angle_conjugated)

assert two_angle_direct(negate_second=False).is_equivalent(conjugated)
print("verified equal:  e^{iα Z0Z1} e^{iβ Z0}|++>  ==  CNOT-conjugated factorization")

assert not two_angle_direct(negate_second=True).is_equivalent(conjugated)
print("detected:         negating the second rotation breaks the branch-phase equality")

verified equal:  e^{iα Z0Z1} e^{iβ Z0}|++>  ==  CNOT-conjugated factorization
detected:         negating the second rotation breaks the branch-phase equality


## Summary

- Section 4.1 reduces *all-angles* equivalence of $C_1 e^{i\alpha Z} C_2|0\rangle$ and
  $D_1 e^{i\alpha Z} D_2|0\rangle$ to a single **exact** stabilizer-state equality
  $C_1 Z^a C_2|0\rangle = D_1 Z^a D_2|0\rangle$, with $Z^a$ an `allocate_symbolic_angle()` bit driving
  a conditional $Z$.
- The equality check needs **no special API**: build each state's `phased_action([], outputs)` and
  compare with `PhasedCircuitAction.is_equivalent` (phase-aware) or `is_equivalent_up_to_signs`
  (phase-blind). This is the phased counterpart of how `OutcomeCompleteSimulation` checks phaseless
  equivalence.
- Because the comparison is **exact**, it certifies the relative branch phase, which is precisely what
  distinguishes $e^{+i\alpha Z}$ from $e^{-i\alpha Z}$ and what ordinary stabilizer simulation discards.